# Q2 · 02 BiGRU + additive attention baseline

**Model.** GloVe 6B 300-d embeddings (frozen for the first epoch, then fine-tuned) → 2-layer bidirectional GRU (256 units per direction, packed so padding is never read) → additive attention pooling → dropout (0.3) → linear (5).

Additive (Bahdanau-style) attention pooling scores every GRU state, normalises the scores over the real tokens only and takes the weighted sum:

$$e_t = v^\top \tanh(W h_t + b), \qquad \alpha_t = \frac{\exp e_t}{\sum_{s:\, m_s = 1} \exp e_s}, \qquad c = \sum_t \alpha_t h_t$$

| | |
|---|---|
| Input | the classic pipeline from `01_preprocessing.ipynb`: lowercased spaCy tokens, vocabulary with min_freq 2, at most 200 tokens |
| Loss | cross-entropy |
| Optimiser | Adam, lr 1e-3, gradient clipping at 1.0 |
| Embeddings | frozen in epoch 1, fine-tuned from epoch 2 |
| Batches | 128 reviews, grouped by length |
| Stopping | validation after every epoch, early stopping on validation macro-F1 (patience 3, at most 15 epochs), best epoch kept |
| Seeds | 42 and 43 |

The test split is not loaded in this notebook.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")
BRANCH = "main"   # the branch to clone when this notebook runs outside the repo (e.g. imported into Kaggle)

root = Path.cwd().resolve()
while not (root / "common").is_dir() and root != root.parent:
    root = root.parent
if not (root / "common").is_dir():
    import subprocess
    root = Path.cwd() / "NLP-Group-Assignment-2"
    if not root.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Akshats-git/NLP-Group-Assignment-2.git", str(root)], check=True)
    os.chdir(root / "Question-2")
sys.path[:0] = [str(root), str(root / "Question-2")]

import torch
from common import DEFAULT_SEEDS, get_device

LIMIT = int(os.environ.get("Q2_LIMIT", 0)) or None   # set for a quick run on a random subset
DEVICE = os.environ.get("Q2_DEVICE", "")             # empty picks cuda, then mps, then cpu
print(get_device(DEVICE or None), torch.cuda.device_count(), "GPU(s)", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Data and model

`classic.prepare` tokenises the reviews and loads the vocabulary and GloVe matrix that `01_preprocessing.ipynb` saved, or builds them if they are missing. On Kaggle, attach a GloVe dataset with `glove.6B.300d.txt` to skip the 822 MB download.

In [ ]:
from common import count_parameters, format_count
from q2 import classic
from q2.models import BiGRUAttention

data, vocab, embeddings = classic.prepare(LIMIT, num_proc=os.cpu_count())
print(f"vocabulary {len(vocab):,}, embedding matrix {tuple(embeddings.shape)}")

model = BiGRUAttention(embeddings).eval()
batch = next(iter(classic.make_loader(data["validation"].select(range(4)), batch_size=4, num_workers=0)))
with torch.no_grad():
    logits, alpha = model(batch["ids"], batch["ids_lengths"], return_attention=True)
print(f"ids {tuple(batch['ids'].shape)} -> logits {tuple(logits.shape)}, attention {tuple(alpha.shape)}")
print("attention sums per review:", alpha.sum(-1).tolist())
print(f"parameters: {format_count(count_parameters(model)['total'])} in total, "
      f"{format_count(model.embedding.weight.numel())} of them in the embedding")
del model

## 2. Training runs

Each run writes `outputs/results/<run>/` (run info, history, summary, validation predictions) and keeps its best weights in `outputs/checkpoints/<run>/best.pt`. A run that already has a `summary.json` is skipped, and on Kaggle an attached earlier version of this notebook brings its finished runs back first.

In [ ]:
from common import load_json, results_dir
from q2.kaggle import import_finished_runs
from q2.train_bigru import BiGRUConfig, train

print("imported from earlier versions:", import_finished_runs("bigru_*") or "none")
runs = [BiGRUConfig(seed=seed, limit=LIMIT, epochs=3 if LIMIT else 15, device=DEVICE) for seed in DEFAULT_SEEDS]

summaries = {}
for config in runs:
    done = results_dir(2) / config.run_name / "summary.json"
    if done.exists():
        print(f"{config.run_name}: already trained, loading its summary")
        summaries[config.run_name] = load_json(done)
        continue
    summaries[config.run_name] = train(config)

## 3. Validation results

Mean ± standard deviation over the two seeds, at each run's best epoch. The test-set comparison with RoBERTa comes in the evaluation step.

In [ ]:
from IPython.display import Markdown
from common import History, format_mean_std, markdown_table, save_json
from common.metrics import summarize_runs

summary = summarize_runs([summaries[c.run_name]["val"] for c in runs])
row = {"model": "BiGRU + attention",
       "Accuracy (%)": format_mean_std(summary["accuracy"]["values"], scale=100),
       "Macro-F1 (%)": format_mean_std(summary["macro_f1"]["values"], scale=100),
       "MAE (stars)": format_mean_std(summary["mae"]["values"], digits=3),
       "best epoch": ", ".join(str(summaries[c.run_name]["best_epoch"]) for c in runs)}
if LIMIT is None:
    save_json(summary, results_dir(2) / "bigru_validation_summary.json")
Markdown(markdown_table([row]))

In [ ]:
from common import figures_dir
from common.plotting import plot_history

histories = {f"seed {c.seed}": History.load(results_dir(2) / c.run_name / "history.json") for c in runs}
out = figures_dir(2) / "bigru_curves.png" if LIMIT is None else None
plot_history(histories, metrics=("loss", "macro_f1"), out_path=out, title="BiGRU + attention: training curves")

Epoch 1 is trained with frozen GloVe vectors and the embeddings are fine-tuned from epoch 2, so the step from epoch 1 to 2 shows what unfreezing adds. Train curves (dashed) are averaged over the epoch with dropout on; validation curves (solid) are on the full 5,000-review validation split.